# Analysis of climate impact on house prices

In [17]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import re
from scipy import stats
import geopandas as gpd
import matplotlib.patches as mpatches

## Load and Transform Data

In [ ]:
# Load data into pd DataFrames
natural_disasters_df = pd.read_csv("..\\data\\climate\\FEMA_Disaster_Declarations.csv")
nri_county_df = pd.read_csv("..\\data\\climate\\NRI_Table_Counties.csv")
housing_county_df = pd.read_csv("..\\data\\housing\\Redfin-Housing-Market-By-County.csv")
housing_state_df = pd.read_csv("..\\data\\housing\\Redfin-Housing-Market-By-State.csv")
fips_df = pd.read_csv("..\\data\\geographic\\fips_master_v2.csv")
fips_state_df = pd.read_csv("..\\data\\geographic\\state_fips_master.csv")

nri_county_df = (
    nri_county_df.assign(
        fips = nri_county_df["STCOFIPS"].astype(str).str.zfill(5),
        nri_risk_score = pd.to_numeric(nri_county_df["RISK_SCORE"], errors = "coerce"),
        nri_risk_rating = nri_county_df["RISK_RATNG"],
        nri_risk_rating_date = nri_county_df["NRI_VER"]
    )[["fips", "nri_risk_score", "nri_risk_rating", "nri_risk_rating_date"]]
    .drop_duplicates(subset = ["fips"])
)

C:\Users\kenny\AppData\Local\Temp\ipykernel_67164\2988868623.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nri_county_df.assign(
C:\Users\kenny\AppData\Local\Temp\ipykernel_67164\2988868623.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nri_county_df.assign(
C:\Users\kenny\AppData\Local\Temp\ipykernel_67164\2988868623.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.c

: 

: 

### FIPS code data

#### Cleaning and Preprocessing

In [ ]:
# For FIPS code by county and state

# Strip trailing and leading whitespaces from all values
fips_df = fips_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Convert fips from int to 5-digit str, padded with zeroes to the left
fips_df['fips'] = (
    fips_df['fips']
        .astype('str')
        .str.zfill(5)
)

# Hawaii's county master combines Maui and Kalawao under 15009, but FEMA emits
# a separate 15005 county code for Kalawao. Add a county-level alias from the
# existing Hawaii reference row so disaster joins can resolve both codes.
kalawao_alias = fips_df.loc[fips_df['fips'] == '15009'].copy()
kalawao_alias['fips'] = '15005'
kalawao_alias['county_name'] = 'Kalawao County'
fips_df = pd.concat([fips_df, kalawao_alias], ignore_index=True)

fips_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3221 entries, 0 to 3220
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   fips         3221 non-null   str  
 1   county_name  3170 non-null   str  
 2   state        3221 non-null   str  
 3   state_long   3221 non-null   str  
 4   msa_code     1808 non-null   str  
 5   msa_name     1808 non-null   str  
 6   msa_type     1808 non-null   str  
 7   csa_code     1150 non-null   str  
 8   csa_name     1150 non-null   str  
dtypes: str(9)
memory usage: 412.6 KB


: 

: 

In [ ]:
# For FIPS code by state only

# Strip trailing and leading whitespaces from all values
fips_state_df = fips_state_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Convert fips from int to 5-digit str, padded with zeroes to the left
fips_state_df['fips'] = (
    fips_state_df['fips']
        .astype('str')
        .str.zfill(5)
)

: 

: 

: 

: 

: 

### Natural disasters data

In [ ]:
# Types of natural disasters
natural_disasters_df["incidentType"].unique()

<ArrowStringArray>
[               'Fire',        'Severe Storm', 'Straight-Line Winds',
               'Flood',           'Hurricane',          'Biological',
        'Winter Storm',             'Tornado',      'Tropical Storm',
          'Earthquake',             'Typhoon',           'Snowstorm',
            'Freezing',       'Mud/Landslide',       'Coastal Storm',
               'Other',    'Severe Ice Storm',     'Dam/Levee Break',
   'Volcanic Eruption', 'Tropical Depression',    'Toxic Substances',
            'Chemical',           'Terrorist',             'Drought',
         'Human Cause',      'Fishing Losses',             'Tsunami']
Length: 27, dtype: str

: 

: 

: 

: 

: 

#### Cleaning and Preprocessing

1. Prepare full 5-digit FIPS code from fipsStateCode and fipsCountyCode
2. Join natural disasters data with county FIPS data to get the counties where disasters were declared  
Note: State-wide disasters should match on the state FIPS (XX000) from the FIPS data

In [ ]:
# Strip trailing and leading whitespaces from all values
natural_disasters_df = natural_disasters_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Convert fipsStateCode into a 2-digit string, fipsCountyCode into a 3-digit string
# then concatenate them to build the 5‑digit FIPS
natural_disasters_df['fipsStateCode'] = (
    natural_disasters_df['fipsStateCode']
        .astype('str')
        .str.zfill(2)
)
natural_disasters_df['fipsCountyCode'] = (
    natural_disasters_df['fipsCountyCode']
        .astype('str')
        .str.zfill(3)
)
natural_disasters_df['fips_code_full'] = (
    natural_disasters_df['fipsStateCode'] + natural_disasters_df['fipsCountyCode']
)

# Change columns with date values to datetime type
natural_disasters_df["declarationDate"] = pd.to_datetime(natural_disasters_df["declarationDate"])
natural_disasters_df["incidentBeginDate"] = pd.to_datetime(natural_disasters_df["incidentBeginDate"])
natural_disasters_df["incidentEndDate"] = pd.to_datetime(natural_disasters_df["incidentEndDate"])

# Join natural_disasters_df on fips_df
natural_disasters_df = pd.merge(
    left=natural_disasters_df,
    right=fips_df,
    how='left',
    left_on='fips_code_full',
    right_on='fips'
)

# Drop duplicate columns, rename columns
natural_disasters_df.drop(columns = ["state_x", "fips_code_full"], inplace = True)
natural_disasters_df = natural_disasters_df.rename(columns = {
    "state_y": "state"
})

_merge
both          67082
left_only      2464
right_only        0
Name: count, dtype: int64

: 

: 

: 

: 

: 

Notes on results from joining natural disaster data on FIPS code data:

_merge<br>
both          67082<br>
left_only      2464<br>
right_only        0<br>
Name: count, dtype: int64<br>

The natural disasters that failed to join on fips_df are associated with states: ['MP', 'GU', 'AS', 'PR', 'VI', 'FM', 'MH', 'PW']<br>
Their fipsStateCode / fipsCountyCode do not match those from the fips code master dataset.

In [ ]:
# Select relevant columns from natural_disasters_df
natural_disasters_df = natural_disasters_df[["fips",
                                             "county_name",
                                             "state",
                                             "disasterNumber",
                                             "declarationDate", 
                                             "incidentType", 
                                             "declarationTitle", 
                                             "ihProgramDeclared", 
                                             "iaProgramDeclared", 
                                             "paProgramDeclared", 
                                             "hmProgramDeclared", 
                                             "incidentBeginDate", 
                                             "incidentEndDate", 
                                             "designatedArea", 
                                             "designatedIncidentTypes"]
                                             ]

: 

: 

: 

: 

: 

### Housing data

#### County-level data

Transform housing data
1. Join on FIPS code data to get FIPS code for each county. Extract county name and state abbreviation from "REGION" to join on FIPS code data.

In [ ]:
# Strip trailing and leading whitespaces from all values
housing_county_df = housing_county_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Change "PERIOD_BEGIN" and "PERIOD_END" to datetime
housing_county_df["PERIOD_BEGIN"] = pd.to_datetime(housing_county_df["PERIOD_BEGIN"])
housing_county_df["PERIOD_END"] = pd.to_datetime(housing_county_df["PERIOD_END"])

# Track month of each observation
housing_county_df["MONTH"] = housing_county_df["PERIOD_BEGIN"].dt.to_period("M")

# Keep only the rows whose values pertain to all properties in the area, rather than to a specific property type
housing_county_df = housing_county_df[housing_county_df["PROPERTY_TYPE"] == "All Residential"]

# Helper functions to transform Redfin housing and FIPS code data into a format suitable for join
def clean_strict(text):
    """Standardizes names while PRESERVING 'city' and 'county' to distinguish between county names like Baltimore City/County."""
    if pd.isna(text): return ""
    text = str(text).lower().split(',')[0]
    # Redfin often uses 'City County' for independent cities; standardize to 'City'
    text = text.replace("city county", "city")
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return " ".join(text.split())

def clean_relaxed(text):
    """Standardizes names by REMOVING all geographic suffixes for a broader match."""
    if pd.isna(text): return ""
    text = str(text).lower().split(',')[0]
    suffixes = r'\b(county|parish|city|borough|municipality|census area|and|city county)\b'
    text = re.sub(suffixes, '', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return " ".join(text.split())

# Join Redfin housing on FIPS code data
fips_df['clean_strict'] = fips_df['county_name'].apply(clean_strict)
fips_df['clean_relaxed'] = fips_df['county_name'].apply(clean_relaxed)
fips_df['clean_msa'] = fips_df['msa_name'].apply(clean_relaxed)
fips_df['clean_csa'] = fips_df['csa_name'].apply(clean_relaxed)

housing_county_df['clean_strict'] = housing_county_df['REGION'].apply(clean_strict)
housing_county_df['clean_relaxed'] = housing_county_df['REGION'].apply(clean_relaxed)
housing_county_df['clean_parent_metro'] = housing_county_df['PARENT_METRO_REGION'].apply(clean_relaxed)

fips_cols = ['fips', 'county_name', 'state', 'msa_name', 'csa_name']

# First, perform Strict County Match
stage1 = pd.merge(housing_county_df, fips_df[fips_cols + ['clean_strict']], 
                  left_on=['clean_strict', 'STATE_CODE'], right_on=['clean_strict', 'state'], how='left')

matched = stage1[stage1['fips'].notna()].copy()
unmatched = stage1[stage1['fips'].isna()].copy()
unmatched = unmatched.drop(columns=[c for c in fips_cols if c in unmatched.columns], errors='ignore')

# Next, perform Relaxed County Match (Handles missing suffixes) ---
stage2 = pd.merge(unmatched, fips_df[fips_cols + ['clean_relaxed']], 
                  left_on=['clean_relaxed', 'STATE_CODE'], right_on=['clean_relaxed', 'state'], how='left')

matched = pd.concat([matched, stage2[stage2['fips'].notna()]])
unmatched = stage2[stage2['fips'].isna()].copy().drop(columns=[c for c in fips_cols if c in stage2.columns], errors='ignore')

# Fallback 1: Join on Parent Metro -> MSA
stage3 = pd.merge(unmatched, fips_df[fips_cols + ['clean_msa']], 
                  left_on=['clean_parent_metro', 'STATE_CODE'], right_on=['clean_msa', 'state'], how='left')

matched = pd.concat([matched, stage3[stage3['fips'].notna()]])
unmatched = stage3[stage3['fips'].isna()].copy().drop(columns=[c for c in fips_cols if c in stage3.columns], errors='ignore')

# Fallback 2: Join on Parent Metro -> CSA
stage4 = pd.merge(unmatched, fips_df[fips_cols + ['clean_csa']], 
                  left_on=['clean_parent_metro', 'STATE_CODE'], right_on=['clean_csa', 'state'], how='left')

# Final Consolidation
housing_county_df = pd.concat([matched, stage4], ignore_index=True)
housing_county_df.drop(columns=['clean_strict', 'clean_relaxed', 'clean_parent_metro', 'state'], inplace=True, errors='ignore')

: 

: 

: 

: 

: 

In [ ]:
# Add FIPS codes manually for counties that couldn't be matched
housing_county_df.loc[housing_county_df['REGION'] == 'Maui County, HI', 'fips'] = '15009'
housing_county_df.loc[housing_county_df['REGION'] == 'La Salle Parish, LA', 'fips'] = '22059'

: 

: 

: 

: 

: 

In [ ]:
# Select relevant columns from housing_county_df
housing_county_df = housing_county_df[["fips", 
                                       "REGION",
                                       "county_name",
                                       "STATE_CODE",
                                       "MONTH",
                                       "PERIOD_BEGIN",
                                       "PERIOD_END", 
                                       "MEDIAN_PPSF", 
                                       "MEDIAN_PPSF_YOY", 
                                       "MEDIAN_LIST_PPSF", 
                                       "MEDIAN_LIST_PPSF_YOY", 
                                       "HOMES_SOLD", 
                                       "HOMES_SOLD_YOY",
                                       "PENDING_SALES",
                                       "PENDING_SALES_YOY",
                                       "INVENTORY",
                                       "INVENTORY_YOY",
                                       "MONTHS_OF_SUPPLY",
                                       "MONTHS_OF_SUPPLY_YOY",
                                       "MEDIAN_DOM",           # Days on Market
                                       "MEDIAN_DOM_YOY",
                                       "AVG_SALE_TO_LIST",
                                       "AVG_SALE_TO_LIST_YOY",
                                       "PRICE_DROPS",
                                       "PRICE_DROPS_YOY"
                                       ]
                                       ]

: 

: 

: 

: 

: 

#### State-level data

Transform housing data by state
1. Join on FIPS code data to get FIPS code for each state

In [ ]:
# Strip trailing and leading whitespaces from all values
housing_state_df = housing_state_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Change "PERIOD_BEGIN" and "PERIOD_END" to datetime
housing_state_df["PERIOD_BEGIN"] = pd.to_datetime(housing_state_df["PERIOD_BEGIN"])
housing_state_df["PERIOD_END"] = pd.to_datetime(housing_state_df["PERIOD_END"])

# Track month of each observation
housing_state_df["MONTH"] = housing_state_df["PERIOD_BEGIN"].dt.to_period("M")

# Keep only the rows whose values pertain to all properties in the area, rather than to a specific property type
housing_state_df = housing_state_df[housing_state_df["PROPERTY_TYPE"] == "All Residential"]

# Clean data in housing_state_df and fips_state_df before join
# Convert all state names to upper case
fips_state_df["state"] = fips_state_df["state"].str.upper()
housing_state_df["REGION"] = housing_state_df["REGION"].str.upper()

# Join Redfin housing on FIPS code data
housing_state_df = pd.merge(housing_state_df, 
                            fips_state_df, 
                            left_on='REGION',
                            right_on='state',
                            how='left')

: 

: 

: 

: 

: 

In [ ]:
# Select columns
housing_state_df["county_name"] = np.nan     # To align with housing data at county level
housing_state_df = housing_state_df[["fips", 
                                       "REGION",
                                       "county_name",
                                       "STATE_CODE",
                                       "MONTH",
                                       "PERIOD_BEGIN",
                                       "PERIOD_END", 
                                       "MEDIAN_PPSF", 
                                       "MEDIAN_PPSF_YOY", 
                                       "MEDIAN_LIST_PPSF", 
                                       "MEDIAN_LIST_PPSF_YOY", 
                                       "HOMES_SOLD", 
                                       "HOMES_SOLD_YOY",
                                       "PENDING_SALES",
                                       "PENDING_SALES_YOY",
                                       "INVENTORY",
                                       "INVENTORY_YOY",
                                       "MONTHS_OF_SUPPLY",
                                       "MONTHS_OF_SUPPLY_YOY",
                                       "MEDIAN_DOM",           # Days on Market
                                       "MEDIAN_DOM_YOY",
                                       "AVG_SALE_TO_LIST",
                                       "AVG_SALE_TO_LIST_YOY",
                                       "PRICE_DROPS",
                                       "PRICE_DROPS_YOY"
                                       ]]

: 

: 

: 

: 

: 

In [ ]:
# Append the housing data by state to the housing data by county
# So that there are rows pertaining to specific counties and states
housing_df = pd.concat([housing_county_df, housing_state_df], ignore_index=True)

: 

: 

: 

: 

: 

#### Measure change in velocities (YOY change) over months

In [ ]:
# Calculate change in YOY values for each month
housing_df = housing_df.sort_values(["fips", "MONTH"])
housing_df["MEDIAN_PPSF_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_PPSF_YOY"].diff()
housing_df["MEDIAN_LIST_PPSF_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_LIST_PPSF_YOY"].diff()
housing_df["HOMES_SOLD_YOY_MOM"] = housing_df.groupby("fips")["HOMES_SOLD_YOY"].diff()
housing_df["PENDING_SALES_YOY_MOM"] = housing_df.groupby("fips")["PENDING_SALES_YOY"].diff()
housing_df["INVENTORY_YOY_MOM"] = housing_df.groupby("fips")["INVENTORY_YOY"].diff()
housing_df["MONTHS_OF_SUPPLY_YOY_MOM"] = housing_df.groupby("fips")["MONTHS_OF_SUPPLY_YOY"].diff()
housing_df["MEDIAN_DOM_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_DOM_YOY"].diff()
housing_df["AVG_SALE_TO_LIST_YOY_MOM"] = housing_df.groupby("fips")["AVG_SALE_TO_LIST_YOY"].diff()

: 

: 

: 

: 

: 

## Time Window Analysis of Natural Disaster Impact

1. Select subset DataFrame of housing data that falls within the window of *n* months before and after incidents of a given type (e.g. fire, flood)
First, filter natural disasters data on given type of incident. 
Then, select only the housing data rows for counties and states where that type of incident occurred and for the months that fall within the *n*-month before and after window of the incident.  
Transformation steps:
- Impute missing `incidentEndDate` values with the estimated end date based on the median duration of that incident type
- Get the month periods for natural disasters and housing data to determine which periods fall within the incident window
- Measure the monthly change in YOY values

2. Plot values of given housing metric over incident window
Select type of plot: Line graph or map plot
For map plot, we compare the mean value of the 6-month-before period against the 6-month-after priod for each county

### Functions

In [ ]:
'''
Get DataFrame of housing data that falls within incident window
'''

def get_housing_data_in_window(incident_type, months = 12):

    # Return error if incident_type is invalid
    if (natural_disasters_df["incidentType"] == incident_type).any() is False:
        return f'incident_type is not valid.'

    # Select data for given incident type from natural disasters data from the last 10 years
    incident_df = natural_disasters_df[(natural_disasters_df["incidentType"] == incident_type) & (natural_disasters_df["incidentBeginDate"] >= "2016-01-01")]

    # For NaT values in incidentEndDate, impute an end date based on the median duration of incident_type

    # Calculate median duration
    incident_df["incident_duration"] = incident_df["incidentEndDate"] - incident_df["incidentBeginDate"]
    incident_median_duration = incident_df[incident_df["incident_duration"].notna()]["incident_duration"].median()

    # Estimate incidentEndDate values for NaTs
    incident_df["incidentEndDate"] = incident_df["incidentEndDate"].fillna(incident_df["incidentBeginDate"] + incident_median_duration)

    # Get month periods that mark beginning and end of incidents for computation
    incident_df["incident_begin_month"] = incident_df["incidentBeginDate"].dt.to_period("M")
    incident_df["incident_end_month"] = incident_df["incidentEndDate"].dt.to_period("M")

    results = []
    incident_num = 1

    for _, event in incident_df.iterrows():
        fips = str(event["fips"]).zfill(5)
        event_start = event["incident_begin_month"]
        event_end = event["incident_end_month"]

        if fips.endswith("000"):
            state_prefix = fips[:2]
            affected_prices = housing_df[
                housing_df["fips"].astype(str).str.zfill(5).str.startswith(state_prefix) &
                ~housing_df["fips"].astype(str).str.zfill(5).str.endswith("000")
            ].copy()
        else:
            affected_prices = housing_df[
                housing_df["fips"].astype(str).str.zfill(5) == fips
            ].copy()

        # The n-month-before and -after windows
        window_periods = [event_start + offset for offset in range((-1 * months), 0)] + [event_end + offset for offset in range(1, (months + 1))]

        # Grab county housing rows within the incident window.
        # State-level incidents apply to all counties in that state.
        county_prices = affected_prices[
            affected_prices["MONTH"].isin(window_periods)
        ].copy()

        if county_prices.empty:
            continue

        # Calculate month offset relative to the event (negative = before, positive = after)
        def get_month_offset(x):
            if x > event_end:
                return int(x.ordinal - event_end.ordinal)
            elif x < event_start:
                return int(x.ordinal - event_start.ordinal)  # Will be negative since x.ordinal < event_start.ordinal
            else:
                return None
            
        county_prices["month_offset_from_incident"] = county_prices["MONTH"].map(get_month_offset)
        county_prices["incident_num"] = incident_num
        results.append(county_prices)
        incident_num += 1
    incident_housing_df = pd.concat(results, ignore_index=True)
    incident_housing_df = incident_housing_df.merge(nri_county_df, on = "fips", how = "left")
    
    return incident_housing_df

: 

: 

: 

: 

: 

In [ ]:
# Generate the incident subsets, save them beside the web visualization, and serve the maps locally
from functools import partial
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from pathlib import Path
import socket
import threading

def get_housing_data_in_window2(incident_type = "Fire", months_before = 12, months_after = 24):
    if months_before < 1 or months_after < 1:
        raise ValueError("months_before and months_after must both be positive integers.")

    if (natural_disasters_df["incidentType"] == incident_type).any() is False:
        raise ValueError("incident_type is not valid.")

    incident_df = natural_disasters_df[
        (natural_disasters_df["incidentType"] == incident_type) &
        (natural_disasters_df["incidentBeginDate"] >= "2016-01-01")
    ].copy()

    incident_df["incident_duration"] = incident_df["incidentEndDate"] - incident_df["incidentBeginDate"]
    incident_median_duration = incident_df[incident_df["incident_duration"].notna()]["incident_duration"].median()
    incident_df["incidentEndDate"] = incident_df["incidentEndDate"].fillna(
        incident_df["incidentBeginDate"] + incident_median_duration
    )

    incident_df["incident_begin_month"] = incident_df["incidentBeginDate"].dt.to_period("M")
    incident_df["incident_end_month"] = incident_df["incidentEndDate"].dt.to_period("M")

    results = []
    incident_num = 1

    for _, event in incident_df.iterrows():
        fips = str(event["fips"]).zfill(5)
        event_start = event["incident_begin_month"]
        event_end = event["incident_end_month"]

        if fips.endswith("000"):
            state_prefix = fips[:2]
            county_prices = housing_df[
                housing_df["fips"].astype(str).str.zfill(5).str.startswith(state_prefix) &
                ~housing_df["fips"].astype(str).str.zfill(5).str.endswith("000")
            ].copy()
        else:
            county_prices = housing_df[
                housing_df["fips"].astype(str).str.zfill(5) == fips
            ].copy()
        if county_prices.empty:
            continue

        available_months = set(county_prices["MONTH"])
        required_after_end_months = {event_end + offset for offset in range(1, months_after + 1)}
        if not required_after_end_months.issubset(available_months):
            continue

        window_periods = [event_start + offset for offset in range(-months_before, 0)]
        window_periods += [event_start + offset for offset in range(1, months_after + 1)]

        county_prices = county_prices[county_prices["MONTH"].isin(window_periods)].copy()
        if county_prices.empty:
            continue

        def get_month_offset(x):
            if x > event_start:
                return int(x.ordinal - event_start.ordinal)
            if x < event_start:
                return int(x.ordinal - event_start.ordinal)
            return None

        county_prices["month_offset_from_incident"] = county_prices["MONTH"].map(get_month_offset)
        county_prices = county_prices[county_prices["month_offset_from_incident"].notna()].copy()
        county_prices["month_offset_from_incident"] = county_prices["month_offset_from_incident"].astype(int)
        county_prices["incident_num"] = incident_num
        results.append(county_prices)
        incident_num += 1

    if not results:
        incident_housing_df2 = housing_df.iloc[0:0].copy()
        incident_housing_df2["month_offset_from_incident"] = pd.Series(dtype = "Int64")
        incident_housing_df2["incident_num"] = pd.Series(dtype = "Int64")
    else:
        incident_housing_df2 = pd.concat(results, ignore_index = True)

    incident_housing_df2 = incident_housing_df2.merge(nri_county_df, on = "fips", how = "left")
    return incident_housing_df2

def _find_open_port(host = "127.0.0.1", start = 8000, end = 8100):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex((host, port)) != 0:
                return port
    raise RuntimeError("Could not find an open port for the local visualization server.")

def launch_bivariate_map(incident_type = "Fire", months = 12, host = "127.0.0.1", port = None):
    output_dir = Path("..") / "output" / "visualizations"
    output_dir.mkdir(parents = True, exist_ok = True)

    incident_housing_df = get_housing_data_in_window(incident_type = incident_type, months = months)
    incident_housing_csv_path = output_dir / "incident_housing.csv"
    incident_housing_df.to_csv(incident_housing_csv_path, index = False)

    incident_housing_df2 = get_housing_data_in_window2(
        incident_type = incident_type,
        months_before = 12,
        months_after = 24,
    )
    incident_housing_24mths_csv_path = output_dir / "incident_housing_24mths.csv"
    incident_housing_df2.to_csv(incident_housing_24mths_csv_path, index = False)


    html_path = output_dir / "index.html"
    if not html_path.exists():
        raise FileNotFoundError(f"Visualization file not found: {html_path.resolve()}")

    existing_server = globals().get("_bivariate_map_server")
    if existing_server is not None:
        existing_server.shutdown()
        existing_server.server_close()

    selected_port = port or _find_open_port(host = host)
    handler = partial(SimpleHTTPRequestHandler, directory = str(output_dir.resolve()))
    httpd = ThreadingHTTPServer((host, selected_port), handler)
    server_thread = threading.Thread(target = httpd.serve_forever, daemon = True)
    server_thread.start()

    map_url = f"http://{host}:{selected_port}/index.html"
    complete_case_map_url = f"{map_url}?complete_cases=1"

    globals()["_bivariate_map_server"] = httpd
    globals()["_bivariate_map_url"] = map_url
    globals()["_bivariate_map_complete_case_url"] = complete_case_map_url
    globals()["_incident_housing_df2"] = incident_housing_df2

    print(f"Saved filtered incident housing data to: {incident_housing_csv_path.resolve()}")
    print(f"Saved 24-month incident housing data to: {incident_housing_24mths_csv_path.resolve()}")
    print(f"Open the bivariate map at: {map_url}")
    print(f"Open the complete-case bivariate map at: {complete_case_map_url}")

    return incident_housing_df, {
        "default": map_url,
        "complete_case": complete_case_map_url,
    }

incident_housing_df, bivariate_map_urls = launch_bivariate_map(incident_type = "Fire", months = 12)


Saved filtered incident housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\incident_housing.csv
Saved 24-month incident housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\incident_housing_24mths.csv
Open the bivariate map at: http://127.0.0.1:8000/index.html
Open the complete-case bivariate map at: http://127.0.0.1:8000/index.html?complete_cases=1


127.0.0.1 - - [26/Mar/2026 21:54:19] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 21:54:19] "GET /incident_housing.csv HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 21:54:19] "GET /incident_housing_24mths.csv HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 22:22:17] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 22:22:17] "GET /incident_housing.csv HTTP/1.1" 304 -
127.0.0.1 - - [26/Mar/2026 22:22:17] "GET /incident_housing_24mths.csv HTTP/1.1" 304 -
127.0.0.1 - - [26/Mar/2026 22:24:55] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 22:27:14] "GET /index.html HTTP/1.1" 304 -
127.0.0.1 - - [26/Mar/2026 22:27:15] "GET /incident_housing.csv HTTP/1.1" 304 -
127.0.0.1 - - [26/Mar/2026 22:27:15] "GET /incident_housing_24mths.csv HTTP/1.1" 304 -
127.0.0.1 - - [26/Mar/2026 22:29:30] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 22:58:02] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [26/Mar/2026 22:58:02] "GET /incident_housing.csv HTTP/1.1" 304 -

: 

: 

: 

: 

: 

#### Statistical Tests

Statistical tests to determine if the values before and after the incident are significantly different.

In [ ]:
# Difference in means of 6-month-before and 6-month-after time periods


: 

: 

: 

: 

: 

In [ ]:
# Linear Mixed Model
import statsmodels.formula.api as smf


: 

: 

: 

: 

: 